In [ ]:
#| default_exp tools

# tools

> The same jobs, shaped for a model to call: strings in, small JSON out, nothing destructive by default.

`rishi` builds a tool schema from a function's signature and docstring, so these are plain functions
with type hints and one-line docs. Hand the list over and a local model can find a classifier, run
it over a folder and tidy the folder up.

```python
from rishi import Chat
from anya.tools import TOOLS
chat = Chat(tools=TOOLS)
chat('Sort ~/Pictures/birds into folders by species.')
```

Two rules shape everything here. Results are capped, because a folder of 2000 photos must not
arrive in a context window one line at a time. And anything that touches files takes `apply=False`
by default, so the first call returns a plan.

In [ ]:
#| export
from __future__ import annotations
import json, sys
from pathlib import Path

import numpy as np
from fastcore.all import AttrDict, L

from anya.core import Preds, arrange, items, load_model
from anya.tasks import (as_model, bench, classify, detect, find_similar, save_masks, segment,
                        sort_images, summarize)

In [ ]:
#| hide
from fastcore.test import *
from tempfile import mkdtemp
from PIL import Image
FIX = Path('fixtures')

## Finding and inspecting a model

In [ ]:
#| export
def find_model(query:str,           # what the model should do, in words
               task:str=None,       # 'classify', 'detect', 'segment', 'embed'
               runtime:str=None,    # 'litert' for on-device, 'onnx', 'coreml'
               n:int=5              # how many candidates
              ) -> dict:
    'Search the model hub for a model anya can run, most downloaded first.'
    from anya.hub import find_models, web_models
    try: ms = find_models(query, task=task, runtime=runtime, n=n)
    except Exception as e:
        try: ms = web_models(query, n=n)
        except Exception: return dict(query=query, error=f'{type(e).__name__}: {e}', models=[])
    return dict(query=query, task=task, runtime=runtime, models=[dict(m) for m in ms])

def model_info(model:str,           # a path, hub repo id, or alias
               task:str=None,       # force the task, when the output shapes are ambiguous
               labels:str=None      # a labels file, when the model does not carry its own
              ) -> dict:
    'Load a model and report what it does: task, runtime, input size, and how many classes.'
    m = as_model(model, **{k: v for k, v in dict(task=task, labels=labels).items() if v})
    return dict(model=m.name, runtime=m.runtime, task=m.task, prep=repr(m.prep),
                n_labels=len(m.labels) if m.labels else 0,
                labels=list(m.labels[:20]) if m.labels else [],
                inputs=[dict(i) for i in m.inputs], outputs=[dict(o) for o in m.outputs],
                max_batch=m.max_bs)

def name_model(alias:str,           # the short name to use from now on
               repo:str,            # the hub repo id or local path it stands for
               file:str=None,       # which file inside the repo, when it ships several
               labels:str=None      # a labels file, when the model does not carry its own
              ) -> dict:
    'Save a short name for a model, so later calls can use it instead of the repo id.'
    from anya.hub import alias as _alias
    kw = {k: v for k, v in dict(file=file, labels=labels).items() if v}
    return dict(alias=alias, saved=_alias(alias, repo, **kw).get(alias))

In [ ]:
#| hide
_i = model_info(str(FIX/'tiny_cls_labels.tflite'))
test_eq(_i['task'], 'classify'); test_eq(_i['runtime'], 'litert'); test_eq(_i['n_labels'], 4)
test_eq(json.loads(json.dumps(_i))['labels'][0], 'red')

## Running one

In [ ]:
#| export
def classify_image(path:str,        # the picture to look at
                   model:str,       # a path, hub repo id, or alias
                   topk:int=3       # how many labels to return
                  ) -> dict:
    'What is in this picture, as ranked labels.'
    p = classify(path, model, topk=topk)
    return dict(src=str(path), model=p.get('model'), labels=p.get('preds', []), error=p.get('error'))

def detect_image(path:str,          # the picture to look at
                 model:str,         # a detector
                 conf:float=0.25,   # ignore anything less confident
                 limit:int=50       # cap the objects returned
                ) -> dict:
    'What objects are in this picture and where, as xyxy boxes in the original pixels.'
    p = detect(path, model, conf=conf)
    o = p.get('objects', [])
    return dict(src=str(path), model=p.get('model'), n=len(o), objects=o[:limit], error=p.get('error'))

def segment_image(path:str,         # the picture to look at
                  model:str         # a segmentation model
                 ) -> dict:
    'Which classes cover this picture, and what share of the pixels each one covers.'
    p = segment(path, model)
    return dict(src=str(path), model=p.get('model'), classes=p.get('classes', []),
                shape=p.get('shape'), error=p.get('error'))

def segment_masks(path:str,         # the picture to look at
                  model:str,        # a segmentation model
                  want:str=None,    # only classes answering to this name, such as 'car'
                  dest:str=None     # where the PNGs go; a `masks` folder beside the picture by default
                 ) -> dict:
    'Write one PNG mask per class, and say where each went. How a mask reaches an editing tool.'
    p = segment(path, model)
    if p.get('error'): return dict(src=str(path), error=p['error'])
    d = dest or str(Path(path).expanduser().parent/'masks')
    return dict(src=str(path), model=p.get('model'), dest=d, shape=p.get('shape'),
                classes=p.get('classes', []), masks=save_masks(p, d, want=want))

In [ ]:
#| hide
_d = Path(mkdtemp())
for i, c in enumerate(['red','green','blue']):
    a = np.zeros((16,16,3), np.uint8); a[...,i] = 255
    Image.fromarray(a).save(_d/f'{c}.png')
(_d/'broken.png').write_bytes(b'nope')

_r = classify_image(str(_d/'green.png'), str(FIX/'tiny_cls_labels.tflite'))
test_eq(_r['labels'][0]['label'], 'green')
test_eq(json.dumps(_r)[:1] , '{')                  # a tool result has to survive json.dumps
_r = detect_image(str(_d/'red.png'), str(FIX/'tiny_det.onnx'))
test_eq(_r['n'], 1)
test_eq(segment_image(str(_d/'green.png'), str(FIX/'tiny_seg.onnx'))['classes'][0]['index'], 1)
_mk = mkdtemp()                                              # not under `_d`: a folder run would read them back
_sm = segment_masks(str(_d/'green.png'), str(FIX/'tiny_seg.onnx'), dest=_mk)
test_eq(list(_sm['masks']), ['class_1'])                     # no labels on this fixture, so index names
test_eq(segment_masks(str(_d/'green.png'), str(FIX/'tiny_seg.onnx'), want='nothing', dest=_mk)['masks'], {})
test_eq(Image.open(_sm['masks']['class_1']).size, (16,16))    # a mask is the size of the picture it came from

## Running a folder

`classify_folder` returns the tally, not the list: 2000 rows is not an answer. `limit` raises the
number of individual items included when the caller does want them, and `save=` writes the whole
thing to a JSON file the caller can read back in pieces.

In [ ]:
#| export
def classify_folder(folder:str,             # folder of pictures
                    model:str,              # a classifier
                    min_score:float=0.0,    # ignore predictions below this
                    limit:int=25,           # how many individual items to include
                    save:str=None           # write every result to this JSON file
                   ) -> dict:
    'Classify every picture in a folder and report the tally.'
    ps = classify(folder, model, topk=1)
    keep = ps.above(min_score) if min_score else ps.ok
    out = dict(folder=str(folder), model=(ps[0].get('model') if len(ps) else None),
               **summarize(ps), kept=len(keep),
               items=[dict(src=p['src'], label=p.label, score=p.score) for p in keep[:limit]])
    if save: out['saved'] = str(ps.save(save))
    return out

def sort_folder(folder:str,                 # folder of pictures to sort
                model:str,                  # a classifier
                dest:str=None,              # where the sorted tree goes; default is <folder>/sorted
                min_score:float=0.5,        # anything less confident goes to unsorted/
                how:str='copy',             # 'copy', 'move', or 'link'
                apply:bool=False,           # False returns the plan and moves nothing
                limit:int=25                # how many planned moves to show
               ) -> dict:
    'Sort a folder of pictures into folders named after what the model saw. Plans unless `apply=True`.'
    r = sort_images(folder, model, dest=dest, min_score=min_score, how=how, dry_run=not apply)
    return dict(folder=str(folder), dest=r['dest'], how=how, applied=bool(apply), n=r['n'],
                moved=r['moved'], labels=r['labels'], counts=r['counts'], failed=r['failed'],
                plan=r['plan'][:limit])

def similar_images(query:str,               # the picture to match
                   folder:str,              # where to look
                   model:str,               # an embedding model
                   n:int=10                 # how many to return
                  ) -> dict:
    'Rank the pictures in a folder by how much they look like one picture.'
    hits = find_similar(query, folder, model, n=n)
    return dict(query=str(query), folder=str(folder), n=len(hits), hits=[dict(h) for h in hits])

def label_video(path:str,                   # the video to look at
                model:str,                  # a classifier or detector
                every:float=1.0,            # seconds between sampled frames
                max_frames:int=20           # stop after this many frames
               ) -> dict:
    'Sample frames from a video and label each one, with its timestamp.'
    m = as_model(model)
    ps = m.predict_video(path, every=every, max_frames=max_frames)
    return dict(src=str(path), model=m.name, task=m.task, n=len(ps), counts=ps.counts(),
                frames=[dict(t=p.get('t'), label=p.label, score=p.score) for p in ps])

def count_images(folder:str,                # folder to look in
                 types:str='image'          # 'image', 'audio', 'video', or 'any'
                ) -> dict:
    'How many files anya would run on in this folder, before running anything.'
    xs = items(folder, types=types)
    return dict(folder=str(folder), n=len(xs), sample=[str(x) for x in xs[:10]])

In [ ]:
#| hide
_cf = classify_folder(str(_d), str(FIX/'tiny_cls_labels.tflite'))
test_eq(_cf['counts'], {'blue': 1, 'green': 1, 'red': 1})
test_eq(_cf['failed'], 1); test_eq(len(_cf['items']), 3)
test_eq(json.loads(json.dumps(_cf))['n'], 4)

_sf = sort_folder(str(_d), str(FIX/'tiny_cls_labels.tflite'))
test_eq(_sf['applied'], False); test_eq(_sf['moved'], 0); test_eq((_d/'sorted').exists(), False)
_sf = sort_folder(str(_d), str(FIX/'tiny_cls_labels.tflite'), apply=True)
test_eq(_sf['moved'], 4); test_eq((_d/'sorted'/'green').exists(), True)

test_eq(count_images(str(_d))['n'] > 0, True)
test_eq(similar_images(str(_d/'red.png'), str(_d), str(FIX/'tiny_emb.onnx'), n=1)['n'], 1)

## The list

`TOOLS` is what you hand to a chat. `SAFE` leaves out anything that writes: the right list for a
model you are not watching.

In [ ]:
#| export
SAFE = [find_model, model_info, count_images, classify_image, detect_image, segment_image,
        classify_folder, similar_images, label_video]
WRITE = [sort_folder, name_model, segment_masks]
TOOLS = SAFE + WRITE

def tool_names() -> list:
    'The names a chat will see.'
    return [f.__name__ for f in TOOLS]

In [ ]:
#| hide
test_eq(len(TOOLS), 12)
test_eq('sort_folder' in tool_names(), True)
test_eq('sort_folder' in [f.__name__ for f in SAFE], False)
# every tool must render as a schema, which is what rishi does with them: no **kw, and every
# parameter annotated and commented
from fastcore.funccall import get_schema
for f in TOOLS:
    sc = get_schema(f, pname='parameters')
    test_eq(bool(sc['description']), True)
    for n, p in sc['parameters']['properties'].items(): test_eq(bool(p.get('description')), True)

## From a shell

`anya <tool> [args]` runs the same functions and prints JSON, which is also the easiest way to try
one without writing a script.

```sh
anya model_info litert-community/some-classifier
anya classify_folder ~/Pictures/birds --model=aussie-birds
anya sort_folder ~/Pictures/birds --model=aussie-birds --apply=1
```

In [ ]:
#| export
def _coerce(v:str, ann):
    'Command line strings into the types a tool declares.'
    if ann is bool: return str(v).lower() not in ('0', 'false', 'no', '')
    if ann is int: return int(v)
    if ann is float: return float(v)
    return v

def main(argv=None) -> int:
    'Entry point for the `anya` command: `anya <tool> [positional…] [--flag=value…]`.'
    import inspect
    argv = list(sys.argv[1:] if argv is None else argv)
    by_name = {f.__name__: f for f in TOOLS}
    if not argv or argv[0] in ('-h', '--help', 'help'):
        print('anya <tool> [args] [--flag=value]\n\ntools:')
        for f in TOOLS: print(f'  {f.__name__:16s} {(f.__doc__ or "").splitlines()[0]}')
        return 0
    name, rest = argv[0], argv[1:]
    if name not in by_name:
        print(f'unknown tool {name!r}; one of: {", ".join(by_name)}', file=sys.stderr); return 2
    f = by_name[name]
    ps = list(inspect.signature(f).parameters.values())
    pos = [a for a in rest if not a.startswith('--')]
    kw = dict(a[2:].split('=', 1) for a in rest if a.startswith('--') and '=' in a)
    kw |= {a[2:]: 'true' for a in rest if a.startswith('--') and '=' not in a}
    args = {p.name: _coerce(v, p.annotation) for p, v in zip(ps, pos)}
    args |= {k: _coerce(v, next((p.annotation for p in ps if p.name == k), str)) for k, v in kw.items()}
    print(json.dumps(f(**args), indent=1, default=str))
    return 0

In [ ]:
#| hide
test_eq(main(['--help']), 0)
test_eq(main(['nope']), 2)
test_eq(main(['count_images', str(_d)]), 0)
test_eq(_coerce('0', bool), False); test_eq(_coerce('3', int), 3)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()